In [18]:
import json
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO


In [19]:
COCO_DIR    = Path("./coco")
VAL_IMG_DIR = COCO_DIR / "images" / "val2017"
ANN_FILE    = COCO_DIR / "annotations" / "instances_val2017.json"

IMG_SIZE    = 640
DEVICE      = 0              # GPU
CONF        = 0.001
IOU         = 0.7            # ✅ REQUIRED (YOLOv12 uses NMS)

WARMUP      = 3
N_IMAGES    = None
RESULTS_CSV = "results_yolov12.csv"

MODELS = [
    "yolo12n.pt",
    "yolo12s.pt",
    "yolo12m.pt",
    "yolo12l.pt",
    "yolo12x.pt",
]


In [20]:
# ── DOWNLOAD ──────────────────────────────────────────────────────────────────
def download_coco():
    COCO_DIR.mkdir(parents=True, exist_ok=True)

    files = {
        "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
        "annotations_trainval2017.zip":
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    }

    for fname, url in files.items():
        dest = COCO_DIR / fname
        if not dest.exists():
            print(f"Downloading {fname} ...")
            urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest) as z:
                z.extractall(COCO_DIR)

    raw = COCO_DIR / "val2017"
    if raw.exists() and not VAL_IMG_DIR.exists():
        VAL_IMG_DIR.parent.mkdir(parents=True, exist_ok=True)
        raw.rename(VAL_IMG_DIR)

In [21]:
# ── COCO META ────────────────────────────────────────────────────────────────
def load_coco_meta():
    with open(ANN_FILE, encoding="utf-8") as f:
        data = json.load(f)

    cat_ids_sorted = sorted(c["id"] for c in data["categories"])
    id_map = {i: cid for i, cid in enumerate(cat_ids_sorted)}

    img_ids = sorted(img["id"] for img in data["images"])
    if N_IMAGES:
        img_ids = img_ids[:N_IMAGES]

    return img_ids, id_map

In [22]:
# ── COCO EVAL ────────────────────────────────────────────────────────────────
def coco_eval(predictions):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import tempfile, os

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(predictions, f)
        tmp = f.name

    coco_gt = COCO(str(ANN_FILE))
    coco_dt = coco_gt.loadRes(tmp)

    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    os.remove(tmp)

    return round(float(evaluator.stats[1]), 4), round(float(evaluator.stats[0]), 4)

In [23]:
# ── BENCHMARK ────────────────────────────────────────────────────────────────
def benchmark(model_name, img_ids, id_map):
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"{'='*50}")

    model = YOLO(model_name)

    # Warmup
    dummy = str(next(VAL_IMG_DIR.glob("*.jpg")))
    for _ in range(WARMUP):
        model(dummy, imgsz=IMG_SIZE, device=DEVICE, conf=CONF, iou=IOU)

    predictions = []
    latencies = []

    for img_id in tqdm(img_ids):
        img_path = VAL_IMG_DIR / f"{img_id:012d}.jpg"
        if not img_path.exists():
            continue

        t0 = time.perf_counter()

        results = model(
            str(img_path),
            imgsz=IMG_SIZE,
            device=DEVICE,
            conf=CONF,
            iou=IOU,
            verbose=False
        )[0]

        latencies.append((time.perf_counter() - t0) * 1000)

        boxes  = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()
        cls    = results.boxes.cls.cpu().numpy().astype(int)

        for box, score, c in zip(boxes, scores, cls):
            x1, y1, x2, y2 = box
            predictions.append({
                "image_id": img_id,
                "category_id": id_map.get(c, c + 1),
                "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                "score": float(score),
            })

    map50, map5095 = coco_eval(predictions)

    avg_ms = np.mean(latencies)

    return {
        "model": model_name,
        "mAP@0.5": map50,
        "mAP@0.5:0.95": map5095,
        "speed_ms": round(avg_ms, 2),
    }



In [24]:
# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    download_coco()

    img_ids, id_map = load_coco_meta()

    rows = []
    for model_name in MODELS:
        rows.append(benchmark(model_name, img_ids, id_map))

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)

    print("\nFINAL RESULTS")
    print(df)

if __name__ == "__main__":
    main()


Model: yolo12n.pt

image 1/1 c:\Users\admin\PFE\PFE-Obstacle-Detection\coco\images\val2017\000000000139.jpg: 448x640 26 persons, 2 bottles, 1 wine glass, 2 cups, 2 bowls, 79 chairs, 6 couchs, 38 potted plants, 22 dining tables, 9 tvs, 6 remotes, 4 keyboards, 2 microwaves, 5 sinks, 16 refrigerators, 18 books, 5 clocks, 17 vases, 46.5ms
Speed: 1.1ms preprocess, 46.5ms inference, 8.0ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 c:\Users\admin\PFE\PFE-Obstacle-Detection\coco\images\val2017\000000000139.jpg: 448x640 26 persons, 2 bottles, 1 wine glass, 2 cups, 2 bowls, 79 chairs, 6 couchs, 38 potted plants, 22 dining tables, 9 tvs, 6 remotes, 4 keyboards, 2 microwaves, 5 sinks, 16 refrigerators, 18 books, 5 clocks, 17 vases, 10.2ms
Speed: 1.1ms preprocess, 10.2ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 c:\Users\admin\PFE\PFE-Obstacle-Detection\coco\images\val2017\000000000139.jpg: 448x640 26 persons, 2 bottles, 1 wine glass, 2 cups, 2 bow

100%|██████████| 5000/5000 [01:22<00:00, 60.58it/s]


loading annotations into memory...
Done (t=0.31s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.82s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.21s).
Accumulating evaluation results...
DONE (t=3.80s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.398
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.553
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.431
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.195
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.442
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.573
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.314
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.513
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

100%|██████████| 5000/5000 [01:41<00:00, 49.49it/s]


loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.23s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=20.40s).
Accumulating evaluation results...
DONE (t=3.11s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.468
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.633
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.507
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.276
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.646
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.355
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.580
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

...

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.76s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=18.74s).
Accumulating evaluation results...
DONE (t=2.81s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.515
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.682
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.560
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.346
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.568
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.683
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.379
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.619
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

100%|██████████| 5000/5000 [04:06<00:00, 20.29it/s]


loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.97s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=19.54s).
Accumulating evaluation results...
DONE (t=2.92s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.527
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.693
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.574
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.350
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.584
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.687
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.387
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.629
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

100%|██████████| 5000/5000 [07:11<00:00, 11.59it/s]


loading annotations into memory...
Done (t=0.35s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.59s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=16.70s).
Accumulating evaluation results...
DONE (t=2.66s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.541
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.705
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.591
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.384
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.597
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.698
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.391
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.642
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [28]:
styled = (
    df.style
    .set_caption("YOLOv26 — Benchmark Results on COCO val2017")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "speed_ms":      "{:.1f} ms",  # corrected
        # "size_MB":      "{:.1f} MB",  # remove or add if exists
        # "params_M":     "{:.1f} M",   # remove or add if exists
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95"], color="black")
    .highlight_min(subset=["speed_ms"], color="black")  # corrected
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th",
         "props": [("background-color", "#000000"), ("color", "white"),
                   ("font-size", "13px"), ("text-align", "center"), ("padding", "8px")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#0B0B0B")]},
    ])
    .hide(axis="index")
)

display(styled)

model,mAP@0.5,mAP@0.5:0.95,speed_ms
yolo12n.pt,0.5528,0.3985,15.8 ms
yolo12s.pt,0.6334,0.4684,19.6 ms
yolo12m.pt,0.6815,0.5149,34.5 ms
yolo12l.pt,0.6925,0.5270,48.5 ms
yolo12x.pt,0.7051,0.5414,85.4 ms


In [31]:
from ultralytics import YOLO
import pandas as pd
import openpyxl

model = YOLO("yolo12s.pt")  # change to your model/weights

results = model.val(
    data="coco.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
)

# Build per-class dataframe
rows = []
for idx, ap in zip(results.ap_class_index, results.maps):
    rows.append({
        "Class Index": idx,
        "Class Name": results.names[idx],
        "AP50-95": round(float(ap), 4),
    })

df = pd.DataFrame(rows)
df = df.sort_values("AP50-95", ascending=False).reset_index(drop=True)

# Add summary row
summary = pd.DataFrame([{
    "Class Index": "-",
    "Class Name": "OVERALL",
    "AP50-95": round(float(results.box.map), 4),
}])

df = pd.concat([summary, df], ignore_index=True)

# Save
df.to_csv("per_class_map.csv", index=False)
df.to_excel("per_class_map.xlsx", index=False)

print(df.to_string(index=False))
print("\nSaved to per_class_map.csv and per_class_map.xlsx")

Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv12s summary (fused): 159 layers, 9,261,840 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 246.290.5 MB/s, size: 184.4 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\coco\labels\val2017.cache... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 5.6it/s 56.3s<0.2s
                   all       5000      36335      0.713      0.587      0.646      0.479
                person       2693      10777       0.83      0.717      0.813      0.593
               bicycle        149        314      0.691       0.49      0.586      0.365
                   car        535       1918      0.749      0.593      0.679      0.465
            motorcycle        159        367      0.809      0.668      0.762   

In [34]:
from ultralytics import YOLO
import pandas as pd
from IPython.display import display

# ── Vos 20 classes cibles (Table 1) ───────────────────────────────
TARGET_CLASSES = [
    "bench", "bicycle", "billboard", "bookcase", "cabinetry",
    "car", "chair", "dog", "door", "fire hydrant",
    "furniture", "kitchen appliance", "person", "plant", "stairs",
    "stop sign", "table", "traffic light", "tree", "waste container"
]

model = YOLO("yolo12s.pt")

results = model.val(
    data="coco.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
)

# ── Construire la table ────────────────────────────────────────────
rows = []
names = results.names

for idx, ap in zip(results.ap_class_index, results.maps):
    class_name = names[idx].lower()
    if class_name in TARGET_CLASSES:
        rows.append({
            "Class":    names[idx].capitalize(),
            "Accuracy": round(float(ap) * 100, 1),
        })

df = pd.DataFrame(rows).sort_values("Class").reset_index(drop=True)

# Ligne Average
average_row = pd.DataFrame([{
    "Class":    "Average",
    "Accuracy": round(df["Accuracy"].mean(), 1),
}])

df_final = pd.concat([df, average_row], ignore_index=True)

# ── Afficher comme Table 1 dans Jupyter ───────────────────────────
styled = (
    df_final.style
    .set_caption("Table 1 — Accuracy of obstacle detection")
    .set_properties(**{"text-align": "left"}, subset=["Class"])
    .set_properties(**{"text-align": "right"}, subset=["Accuracy"])
    .apply(lambda x: ["font-style: italic; border-top: 1px solid black"] * len(x)
           if x.name == len(df_final) - 1 else [""] * len(x), axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px"), ("text-align", "left")]},
        {"selector": "thead th", "props": [("border-top", "2px solid black"), ("border-bottom", "1px solid black")]},
        {"selector": "tbody tr:last-child td", "props": [("border-top", "1px solid black")]},
        {"selector": "tbody tr:last-child", "props": [("font-style", "italic")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "350px")]},
        {"selector": "td, th", "props": [("padding", "4px 12px"), ("border", "none")]},
    ])
    .hide(axis="index")
)

display(styled)

Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv12s summary (fused): 159 layers, 9,261,840 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 615.3444.7 MB/s, size: 146.5 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\coco\labels\val2017.cache... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 5.4it/s 58.1s<0.2s
                   all       5000      36335      0.713      0.587      0.646      0.479
                person       2693      10777       0.83      0.717      0.813      0.593
               bicycle        149        314      0.691       0.49      0.586      0.365
                   car        535       1918      0.749      0.593      0.679      0.465
            motorcycle        159        367      0.809      0.668      0.762  

Class,Accuracy
Bench,30.300000
Bicycle,36.500000
Car,46.500000
Chair,36.700000
Dog,72.900000
Fire hydrant,74.900000
Person,59.300000
Stop sign,71.300000
Traffic light,29.100000
Average,50.800000
